# Übung 1: Environment Setup & erstes Notebook
## AI Frameworks & Tools -- Block 1

**Ziel**: Sicherstellen, dass alle Bibliotheken korrekt installiert sind, und den uv-Workflow hands-on erleben.

---

## Voraussetzung: Projekt-Setup mit uv

Bevor ihr dieses Notebook ausführt, muss das Projekt eingerichtet sein.  
Öffnet ein **Terminal** und führt folgende Schritte aus:

```bash
# 1. uv installieren (falls noch nicht vorhanden)
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. In den Block1_Material Ordner wechseln (dort liegt die pyproject.toml)
cd Block1_Material

# 3. Virtual Environment erstellen
uv venv

# 4. Environment aktivieren
# source .venv/bin/activate        # Linux / Mac
.venv\Scripts\activate         # Windows

# 5. Alle Dependencies aus pyproject.toml installieren
uv pip install -e ".[dev]"

# 6. JupyterLab starten
jupyter lab
```

Danach könnt ihr dieses Notebook öffnen und alle Zellen ausführen.

---

## Teil A: Installations-Check
Führe die folgenden Zellen aus. Alle Imports müssen fehlerfrei durchlaufen.

In [ ]:
import sys
print(f"Python: {sys.version}")
print(f"Pfad:   {sys.executable}")

In [ ]:
# Kern-Bibliotheken
import numpy as np
import pandas as pd
import polars as pl
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("=== Kern-Bibliotheken ===")
for name, mod in [("NumPy", np), ("Pandas", pd), ("Polars", pl),
                   ("Matplotlib", matplotlib), ("Seaborn", sns),
                   ("Scikit-Learn", sklearn)]:
    print(f"  {name:15s} {mod.__version__}")
print("Alle Kern-Bibliotheken OK!")

In [ ]:
# Deep Learning & NLP
import torch
import transformers

print("=== Deep Learning & NLP ===")
print(f"  {'PyTorch':15s} {torch.__version__}")
print(f"  {'Transformers':15s} {transformers.__version__}")
print(f"  {'CUDA verfügbar':15s} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {'GPU':15s} {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f"  {'Apple MPS':15s} verfügbar")
else:
    print(f"  {'Device':15s} CPU (völlig ausreichend für GPT-2)")
print("Deep Learning Bibliotheken OK")

In [ ]:
# Schnelltest: Kann PyTorch rechnen?
a = torch.randn(3, 3)
b = torch.randn(3, 3)
c = a @ b
print(f"Matrixmultiplikation: {a.shape} @ {b.shape} = {c.shape}")
print(f"Ergebnis:\n{c}")
print("PyTorch funktioniert")

In [ ]:
# Schnelltest: Kann HuggingFace ein Modell laden?
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokens = tokenizer.encode("Hello, AI Frameworks")
decoded = [tokenizer.decode(t) for t in tokens]
print(f"Tokenisierung: 'Hello, AI Frameworks'")
print(f"  Token IDs: {tokens}")
print(f"  Decoded:   {decoded}")
print("\nHuggingFace funktioniert")

---
## Teil B: pyproject.toml inspizieren

Die `pyproject.toml` liegt als Referenzdatei im `Block1_Material/` Ordner bereit.  
Sie enthält alle Dependencies für das gesamte Semester.

In [ ]:
# pyproject.toml anzeigen
import pathlib

candidates = [
    pathlib.Path("../pyproject.toml"),
    pathlib.Path("pyproject.toml"),
]

toml_path = None
for c in candidates:
    if c.exists():
        toml_path = c
        break

if toml_path:
    print(f"Gefunden: {toml_path.resolve()}\n")
    print(toml_path.read_text())
else:
    print("pyproject.toml nicht gefunden.")
    print(f"Aktuelles Verzeichnis: {pathlib.Path.cwd()}")

### Fragen zu pyproject.toml

**Frage 1**: Welche Dependencies sind eingetragen?

*Deine Antwort*: ...

**Frage 2**: Was ist der Unterschied zu einer requirements.txt?

*Deine Antwort*: ...

**Frage 3**: Was würde `uv lock` erzeugen und warum ist das nützlich?

*Deine Antwort*: ...

---
## Teil C: Dependency-Management live erleben

In diesem Teil erlebt ihr den **add → lock → remove**-Zyklus von uv hands-on.

### C.1 Package hinzufügen

Öffne ein **Terminal** (nicht diese Zelle!) und führe aus:
```bash
uv add requests
```

Dann führe die nächste Zelle aus, um zu prüfen, ob es geklappt hat:

In [ ]:
# Prüfe: Ist requests jetzt installiert?
try:
    import requests
    r = requests.get("https://httpbin.org/json")
    print(f"Status: {r.status_code}")
    print(f"Antwort: {r.json()['slideshow']['title']}")
    print("\nrequests funktioniert!")
except ImportError:
    print("requests ist noch nicht installiert.")
    print("Führe im Terminal aus: uv add requests")

### C.2 Was hat sich in pyproject.toml geändert?

Führe die nächste Zelle aus und vergleiche mit dem Output von oben:

In [ ]:
# pyproject.toml nochmal anzeigen -- was ist neu?
if toml_path and toml_path.exists():
    print(toml_path.read_text())
else:
    print("pyproject.toml nicht gefunden.")

**AUFGABE**: Was hat sich geändert? Wo steht `requests` jetzt?

*Deine Antwort*: ...

### C.3 Lock-Datei verstehen

Führe im **Terminal** aus:
```bash
uv lock
```

Dann führe die nächste Zelle aus:

In [ ]:
# Lock-Datei inspizieren
lock_candidates = [
    pathlib.Path("../uv.lock"),
    pathlib.Path("uv.lock"),
]

lock_path = None
for c in lock_candidates:
    if c.exists():
        lock_path = c
        break

if lock_path:
    content = lock_path.read_text()
    # Zähle Packages
    packages = [line for line in content.split('\n') if line.startswith('[[package]]') or line.startswith('name = ')]
    package_names = [line.split('"')[1] for line in content.split('\n') if line.strip().startswith('name = ')]
    print(f"Anzahl Packages in uv.lock: {len(package_names)}")
    print(f"\nPackages: {', '.join(package_names[:20])}")
    if len(package_names) > 20:
        print(f"  ... und {len(package_names) - 20} weitere")
else:
    print("uv.lock nicht gefunden.")
    print("Führe im Terminal aus: uv lock")

**AUFGABE**: Beantworte:

1. Wie viele Packages stehen in `uv.lock` -- warum sind es mehr als in `pyproject.toml`?

   *Deine Antwort*: ...

2. Was passiert, wenn ein Teamkollege `uv sync` auf seinem Rechner ausführt?

   *Deine Antwort*: ...

### C.4 Package wieder entfernen

Führe im **Terminal** aus:
```bash
uv remove requests
```

Prüfe nochmal `pyproject.toml` -- ist `requests` verschwunden?

In [ ]:
# Finale Prüfung: requests sollte nicht mehr in pyproject.toml stehen
if toml_path and toml_path.exists():
    content = toml_path.read_text()
    if 'requests' in content:
        print("requests steht noch in pyproject.toml -- nochmal 'uv remove requests' ausführen!")
    else:
        print("requests erfolgreich entfernt!")
    print("\n" + content)

### C.5 Reflexion

**AUFGABE**: Erkläre in 2-3 Sätzen: Was ist der Unterschied zwischen `pyproject.toml` und `uv.lock`? Warum braucht man beides?

*Deine Antwort*: ...

---
## Teil D: Colab als Fallback

Lade dieses Notebook in Google Colab hoch und führe Teil A aus.  
Diskutiere kurz mit deinem Nachbarn: In welchen Situationen lokal, in welchen Colab?

---
## Zusammenfassung

Wenn alle Zellen oben fehlerfrei durchgelaufen sind, ist euer Environment bereit!

**Nächster Schritt**: Übung 2 -- Explorative Datenanalyse (02_eda.ipynb)